# The Fastest Part Is the Exit
## Phase-Aware Validation of Offensive Line Combine Tracking Across Drills, Movements, and Rookie Pass Protection

**Methodological & Scouting Validation Study (NFL Big Data Bowl 2027).** This notebook
rebuilds the complete offensive line (C/G/T) Combine phase-segmentation audit across
all three OL footwork drills (`Pass-Pro Mirror Drill`, `5-Yard Wave Drill`, and
`Pass Rush Drop` — 470 accepted recordings), two-attempt trial repeatability bounds,
rookie one-on-one pass-protection out-of-time modeling (across binary pressure allowed
and continuous Next Gen Stats peak pressure probability), same-axis early-game lateral
movement transfer (`0–1.5 s`), and 400-replicate full-refit player bootstrap uncertainty.

CPU only; no external data or internet required. Attach the competition dataset and
run all cells. Physical Next Gen Stats tracking units are verified across the pipeline
(`x, y` in `yards`, `s` in `yd/s`, `a` in `yd/s²`, `10 Hz` sampling).


In [ ]:
import os
from pathlib import Path
import platform
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display, Image, Markdown

if os.environ.get('NFL_DATA_DIR'):
    DATA_DIR = Path(os.environ['NFL_DATA_DIR'])
else:
    candidates = [Path.cwd() / 'data/nfl-big-data-bowl-2027',
                  Path.cwd().parent / 'data/nfl-big-data-bowl-2027']
    DATA_DIR = next((p for p in candidates if (p / 'players.csv').exists()), None)
    if DATA_DIR is None and Path('/kaggle/input').exists():
        matches = list(Path('/kaggle/input').rglob('combine_tracking.csv'))
        if len(matches) != 1:
            raise RuntimeError('Attach exactly one competition dataset, or set NFL_DATA_DIR.')
        DATA_DIR = matches[0].parent
    if DATA_DIR is None:
        raise FileNotFoundError('Competition CSVs not found; set NFL_DATA_DIR to their directory.')

WORK_DIR = Path(os.environ.get('NFL_WORK_DIR', '/kaggle/working' if Path('/kaggle').exists()
                            else str(Path.cwd() / 'notebook_output')))
WORK_DIR.mkdir(parents=True, exist_ok=True)
required = ['players.csv', 'combine_results.csv', 'combine_tracking.csv', 'games.csv',
            'player_play.csv'] + [f'game_tracking_{y}.csv' for y in (2023, 2024, 2025)]
assert all((DATA_DIR / f).is_file() for f in required), 'Missing competition files'
print({'python': platform.python_version(), 'numpy': np.__version__,
       'pandas': pd.__version__, 'sklearn': sklearn.__version__,
       'data_directory': str(DATA_DIR), 'output_directory': str(WORK_DIR)})


## 1. Rookie Offensive Line Cohort Metadata

We restrict to offensive linemen (`C`, `G`, `T`) in their rookie regular season on
non-nullified plays. The downstream pass-protection analysis retains `C/G/T` game
alignments, an observed `pressure_allowed` and `peak_pressure_probability_allowed`
label, **exactly one encountered pass rusher**, and at least `100` eligible one-on-one
opportunities per player (`50` players, `20,853` snaps, `1,850` pressures).


In [ ]:
"""Build the required rookie OL metadata directly from competition inputs."""
from pathlib import Path
import pandas as pd

ROOT = WORK_DIR
DATA = DATA_DIR
OUT = ROOT / "reports/trenches"


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    players = pd.read_csv(DATA / "players.csv")
    games = pd.read_csv(DATA / "games.csv")
    columns = ["game_id", "play_id", "nfl_id", "lined_up_position", "play_nullified_by_penalty",
               "pressure_allowed", "peak_pressure_probability_allowed", "time_to_pressure_allowed",
               "pass_rushers_encountered", "dropback_duration", "down", "yards_to_go",
               "offense_formation"]
    plays = pd.read_csv(DATA / "player_play.csv", usecols=columns)
    assert not plays.duplicated(["game_id", "play_id", "nfl_id"]).any()
    plays = plays.merge(players[["nfl_id", "nfl_position", "draft_year"]], on="nfl_id", validate="many_to_one")
    plays = plays.merge(games[["game_id", "season", "season_type"]], on="game_id", validate="many_to_one")
    plays = plays.loc[plays.nfl_position.isin(["T", "G", "C"]) & plays.season_type.eq("REG") &
                      plays.season.eq(plays.draft_year) & plays.play_nullified_by_penalty.eq("N")]
    plays.to_csv(OUT / "ol_rookie_plays.csv", index=False)
    print(f"Rookie OL metadata: {len(plays)} player-plays, {plays.nfl_id.nunique()} players")


main()

## 2. Segment Functional Footwork from the Exit Sprint Across All Three OL Drills

Both spelling variants of the Mirror drill (`PASS_PRO-MIRROR_DRILL` and
`PASS_PRO_MIRROR_DRILL`) are combined. A sustained forward departure of `1.5 yards`
for four consecutive frames (`0.4 s`) relative to the initial five-frame median $x_0$
anchor defines exit onset. We require strictly increasing `10 Hz` timestamps with no
frame gaps above `0.21 s`, at least `3.0 seconds` of mirror tracking, and at least
two detected lateral reversals (`113` accepted players/attempts).

Five-frame quadratic Savitzky–Golay smoothing of $y(t)$ precedes differentiation to
compute lateral velocity $v_y(t)$ (`yd/s`). The two predeclared mirror-phase features
are `lateral_speed_p75` (75th percentile of $|v_y(t)|$ in `yd/s`) and
`reversal_recovery` (median ratio of post-turn to pre-turn lateral speed). We compare
these against `whole_drill_speed` (95th percentile of 3-frame median-smoothed total
speed $s(t)$ across the full recording).

We also audit `FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE` (`118` attempts) and
`PASS_RUSH_DROP` (`239` attempts, including `118` OL with two tracked repetitions)
to test whether exit-sprint dominance is universal across OL Combine drills and to
quantify trial-to-trial repeatability.


In [ ]:
"""Segment OL mirror motion from its exit sprint before outcome analysis."""
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

ROOT = WORK_DIR
DATA = DATA_DIR
OUT = ROOT / "reports/trenches"
FEATURES = ["lateral_speed_p75", "reversal_recovery"]


def measure(g, boundary=1.5):
    g = g.sort_values("time").reset_index(drop=True)
    t = (g.time - g.time.iloc[0]).dt.total_seconds().to_numpy()
    if len(g) < 35 or np.any(np.diff(t) <= 0) or np.max(np.diff(t)) > .21:
        return None, "invalid_time_or_short", None
    x = g.x.to_numpy()
    departure = x - np.median(x[:5]) > boundary
    exits = [i for i in range(len(t) - 3) if departure[i:i + 4].all()]
    if not exits:
        return None, "no_exit", None
    end = exits[0]
    if t[end - 1] < 3:
        return None, "short_mirror", None
    mirror = g.iloc[:end]
    mt = t[:end]
    y = savgol_filter(mirror.y.to_numpy(), 5, 2)
    vy = np.gradient(y, mt)
    candidates = []
    for i in range(5, len(mt) - 6):
        if vy[i - 1] * vy[i] > 0:
            continue
        before = (mt >= mt[i] - .5 - 1e-8) & (mt <= mt[i] - .2 + 1e-8)
        after = (mt >= mt[i] + .3 - 1e-8) & (mt <= mt[i] + .6 + 1e-8)
        if abs(y[i] - y[i - 4]) < .25 or abs(y[i] - y[i + 4]) < .25:
            continue
        entry = np.mean(np.abs(vy[before]))
        if entry < .5:
            continue
        if candidates and mt[i] - mt[candidates[-1][0]] < .6:
            continue
        candidates.append((i, np.mean(np.abs(vy[after])) / entry))
    if len(candidates) < 2:
        return None, "fewer_than_two_reversals", (g, end, candidates, vy)
    smoothed_speed = g.s.rolling(3, center=True, min_periods=1).median().to_numpy()
    global_max_index = int(np.argmax(smoothed_speed))
    record = {"lateral_speed_p75": float(np.quantile(np.abs(vy), .75)),
              "reversal_recovery": float(np.median([ratio for _, ratio in candidates])),
              "whole_drill_speed": float(np.quantile(smoothed_speed, .95)),
              "mirror_duration": float(mt[-1]), "total_duration": float(t[-1]),
              "reversals": len(candidates), "peak_in_exit": bool(global_max_index >= end)}
    return record, None, (g, end, candidates, vy)


def audit_ol_drill_suite(players, raw_tracking):
    """Audit exit-sprint dominance and phase rank correlations across all three OL footwork drills."""
    ol_ids = set(players.loc[players.nfl_position.isin(["T", "G", "C"]), "nfl_id"])
    ol_track = raw_tracking.loc[raw_tracking.entity_type.eq("PLAYER") & raw_tracking.nfl_id.isin(ol_ids)].copy()
    ol_track["time"] = pd.to_datetime(ol_track.time)

    # 1. Five-Yard Wave Drill (slide and shuffle in 5-yd box followed by -x finish sprint)
    wave_rows = []
    wave_sub = ol_track.loc[ol_track.drill_name.eq("FIVE_YARD_WAVE_DRILL_SLIDE_AND_SHUFFLE")]
    for (nid, yr, eid, att), g in wave_sub.groupby(["nfl_id", "draft_year", "event_id", "attempt"]):
        g = g.sort_values("time").reset_index(drop=True)
        t = (g.time - g.time.iloc[0]).dt.total_seconds().to_numpy()
        if len(t) < 50 or np.any(np.diff(t) <= 0) or np.max(np.diff(t)) > .21:
            continue
        x = savgol_filter(g.x.to_numpy(), 5, 2)
        y = savgol_filter(g.y.to_numpy(), 5, 2)
        vy = np.gradient(y, t)
        v2d = np.hypot(np.gradient(x, t), vy)
        s_smooth = g.s.rolling(3, center=True, min_periods=1).median().to_numpy()
        x0 = np.median(x[:5])
        dep = (x0 - x) > 2.0
        exits = [i for i in range(len(t) - 3) if dep[i:i + 4].all()]
        if not exits or t[exits[0]] < 5.0:
            continue
        end = exits[0]
        wave_rows.append({
            "nfl_id": int(nid), "draft_year": int(yr), "event_id": str(eid), "attempt": int(att),
            "wave_lateral_speed_p75": float(np.quantile(np.abs(vy[:end]), .75)),
            "wave_shuffle_speed_p75": float(np.quantile(v2d[:end], .75)),
            "wave_whole_speed_p95": float(np.quantile(s_smooth, .95)),
            "wave_peak_in_exit": bool(int(np.argmax(s_smooth)) >= end),
        })
    wave_df = pd.DataFrame(wave_rows)

    # 2. Pass Rush Drop (diagonal kick-slide drop followed by -x finish sprint; 2 attempts per player)
    drop_rows = []
    drop_sub = ol_track.loc[ol_track.drill_name.eq("PASS_RUSH_DROP")]
    for (nid, yr, eid, att), g in drop_sub.groupby(["nfl_id", "draft_year", "event_id", "attempt"]):
        g = g.sort_values("time").reset_index(drop=True)
        t = (g.time - g.time.iloc[0]).dt.total_seconds().to_numpy()
        if len(t) < 25 or np.any(np.diff(t) <= 0) or np.max(np.diff(t)) > .21:
            continue
        x = savgol_filter(g.x.to_numpy(), 5, 2)
        y = savgol_filter(g.y.to_numpy(), 5, 2)
        vx = np.gradient(x, t)
        vy = np.gradient(y, t)
        v2d = np.hypot(vx, vy)
        s_smooth = g.s.rolling(3, center=True, min_periods=1).median().to_numpy()
        dep = np.abs(x - x[0]) > 2.2
        exits = [i for i in range(len(t) - 3) if dep[i:i + 4].all()]
        if not exits or exits[0] < 8:
            continue
        end = exits[0]
        drop_rows.append({
            "nfl_id": int(nid), "draft_year": int(yr), "event_id": str(eid), "attempt": int(att),
            "drop_lateral_speed_p75": float(np.quantile(np.abs(vy[:end]), .75)),
            "drop_depth_speed_p75": float(np.quantile(np.abs(vx[:end]), .75)),
            "drop_2d_speed_p75": float(np.quantile(v2d[:end], .75)),
            "drop_whole_speed_p95": float(np.quantile(s_smooth, .95)),
            "drop_peak_in_exit": bool(int(np.argmax(s_smooth)) >= end),
        })
    drop_df = pd.DataFrame(drop_rows).sort_values(["nfl_id", "attempt", "event_id"])
    drop_df["rep"] = drop_df.groupby("nfl_id").cumcount()
    drop_piv = drop_df.loc[drop_df.rep.lt(2)].pivot(index="nfl_id", columns="rep")
    repeatability = {}
    for col in ["drop_whole_speed_p95", "drop_lateral_speed_p75", "drop_depth_speed_p75", "drop_2d_speed_p75"]:
        valid = drop_piv[col].dropna()
        repeatability[col] = {
            "paired_players": int(len(valid)),
            "pearson_r": float(valid[0].corr(valid[1])),
            "spearman_r": float(valid[0].corr(valid[1], method="spearman")),
        }
    drop_player = drop_df.groupby("nfl_id")[["drop_lateral_speed_p75", "drop_depth_speed_p75",
                                             "drop_2d_speed_p75", "drop_whole_speed_p95"]].median()
    return wave_df, drop_df, drop_player, repeatability


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    players = pd.read_csv(DATA / "players.csv")
    ol = players.loc[players.nfl_position.isin(["T", "G", "C"])]
    raw_tracking = pd.read_csv(DATA / "combine_tracking.csv")
    tracking = raw_tracking.loc[raw_tracking.entity_type.eq("PLAYER") & raw_tracking.nfl_id.isin(ol.nfl_id) &
                                raw_tracking.drill_name.isin(["PASS_PRO-MIRROR_DRILL", "PASS_PRO_MIRROR_DRILL"])].copy()
    tracking["time"] = pd.to_datetime(tracking.time)
    results, rejected, details = [], [], {}
    sensitivity = []
    for key, g in tracking.groupby(["nfl_id", "draft_year", "event_id", "attempt"]):
        ids = dict(zip(["nfl_id", "draft_year", "event_id", "attempt"], key))
        result, reason, detail = measure(g)
        if reason:
            rejected.append({**ids, "reason": reason})
        else:
            results.append({**ids, **result})
            details[key] = detail
        for boundary in (1.0, 2.0):
            alternative, reason, _ = measure(g, boundary)
            if alternative:
                sensitivity.append({**ids, "boundary": boundary, **alternative})
    attempts = pd.DataFrame(results)
    attempts.to_csv(OUT / "mirror_attempt_features.csv", index=False)
    pd.DataFrame(rejected).to_csv(OUT / "mirror_rejected.csv", index=False)
    pd.DataFrame(sensitivity).to_csv(OUT / "mirror_boundary_sensitivity.csv", index=False)
    feature = attempts.groupby("nfl_id")[FEATURES + ["whole_drill_speed", "mirror_duration", "peak_in_exit"]].median()
    feature.to_csv(OUT / "mirror_player_features.csv")

    wave_df, drop_df, drop_player, drop_repeatability = audit_ol_drill_suite(players, raw_tracking)
    suite_summary = pd.DataFrame([
        {
            "drill": "Pass-Pro Mirror Drill",
            "accepted_attempts": int(len(attempts)),
            "accepted_players": int(attempts.nfl_id.nunique()),
            "peak_in_exit_fraction": float(attempts.peak_in_exit.mean()),
            "phase_vs_whole_spearman": float(attempts.lateral_speed_p75.corr(attempts.whole_drill_speed, method="spearman")),
        },
        {
            "drill": "5-Yard Wave Drill (Slide & Shuffle)",
            "accepted_attempts": int(len(wave_df)),
            "accepted_players": int(wave_df.nfl_id.nunique()),
            "peak_in_exit_fraction": float(wave_df.wave_peak_in_exit.mean()),
            "phase_vs_whole_spearman": float(wave_df.wave_lateral_speed_p75.corr(wave_df.wave_whole_speed_p95, method="spearman")),
        },
        {
            "drill": "Pass Rush Drop (Kick-Slide)",
            "accepted_attempts": int(len(drop_df)),
            "accepted_players": int(drop_df.nfl_id.nunique()),
            "peak_in_exit_fraction": float(drop_df.drop_peak_in_exit.mean()),
            "phase_vs_whole_spearman": float(drop_player.drop_lateral_speed_p75.corr(drop_player.drop_whole_speed_p95, method="spearman")),
        },
    ])
    suite_summary.to_csv(OUT / "ol_drill_suite_audit.csv", index=False)

    audit = {"raw_players": tracking.nfl_id.nunique(), "accepted_players": len(feature),
             "accepted_attempts": len(attempts), "rejected_attempts": len(rejected),
             "peak_in_exit_fraction": float(attempts.peak_in_exit.mean()),
             "by_year": attempts.groupby("draft_year").agg(players=("nfl_id", "nunique"),
                 peak_in_exit_fraction=("peak_in_exit", "mean"), duration=("mirror_duration", "median")).reset_index().to_dict("records"),
             "boundary_rank_correlations": [],
             "ol_drill_suite_audit": suite_summary.to_dict("records"),
             "pass_rush_drop_repeatability": drop_repeatability}
    for boundary in (1.0, 2.0):
        alternative = pd.DataFrame(sensitivity)
        alternative = alternative.loc[alternative.boundary.eq(boundary)]
        paired = attempts.merge(alternative, on=["nfl_id", "draft_year", "event_id", "attempt"], suffixes=("_main", "_alt"))
        audit["boundary_rank_correlations"].append({"boundary": boundary, "paired_attempts": len(paired),
            **{name: float(paired[name + "_main"].corr(paired[name + "_alt"], method="spearman")) for name in FEATURES}})
    # Review all accepted attempts in a multipage PDF; a nine-example PNG is also saved.
    from matplotlib.backends.backend_pdf import PdfPages
    with PdfPages(OUT / "mirror_all_phase_checks.pdf") as pdf:
        entries = list(details.items())
        for offset in range(0, len(entries), 12):
            fig, axes = plt.subplots(3, 4, figsize=(13, 10))
            for ax in axes.flat:
                ax.set_visible(False)
            for ax, (key, (g, end, turns, vy)) in zip(axes.flat, entries[offset:offset + 12]):
                ax.set_visible(True)
                ax.plot(g.x.iloc[:end], g.y.iloc[:end], color="#176b87", label="Mirror")
                ax.plot(g.x.iloc[end - 1:], g.y.iloc[end - 1:], color="#e99b37", label="Exit")
                if turns:
                    idx = [i for i, _ in turns]
                    ax.scatter(g.x.iloc[idx], g.y.iloc[idx], color="#b43b45", s=18)
                ax.set_title(f"{key[1]} {ol.set_index('nfl_id').loc[key[0], 'display_name']}", fontsize=9)
                ax.set_aspect("equal", adjustable="datalim")
            axes.flat[0].legend(fontsize=7)
            fig.suptitle("Mirror phase (blue), exit sprint (orange), detected reversals (red)")
            fig.tight_layout()
            pdf.savefig(fig)
            if offset == 0:
                fig.savefig(OUT / "mirror_phase_examples.png", dpi=140)
            plt.close(fig)
    (OUT / "mirror_feature_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print(json.dumps(audit, indent=2))


main()

## 3. Out-of-Time Validation: Rookie One-on-One Pass Protection

The classic player baseline uses official `40-yard dash`, `10-yard split`,
`combine_weight`, `arm_length`, and position (`C/G/T`). We compare `Classic Combine`
alone, `Classic + Whole-Drill Speed`, and `Classic + Mirror-Phase Features` using
fixed `Ridge(alpha=10)` models under strict forward-year evaluation (`2023 -> 2024`
and `2023–2024 -> 2025`), with training-only imputation, scaling, and play-context
adjustment (`lined_up_position`, `offense_formation`, `down`, `yards_to_go`,
`dropback_duration`, and `duration_sq`).

Three outcome specifications are evaluated side-by-side:
1. **Primary (`primary_with_duration`):** Binary `pressure_allowed` adjusted for play context including dropback duration.
2. **Predeclared Sensitivity (`sensitivity_without_duration`):** Binary `pressure_allowed` without dropback duration.
3. **Continuous NGS Outcome (`continuous_pressure_probability`):** Context-adjusted `peak_pressure_probability_allowed` (where game split-half reliability increases from `r = 0.367` to `r = 0.696`).


In [ ]:
"""Fixed-design mirror phase features versus rookie one-on-one pressure allowed."""
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = WORK_DIR
DATA = DATA_DIR
OUT = ROOT / "reports/trenches"
CLASSIC = ["forty", "ten_yd_split", "combine_weight", "arm_length"]
MIRROR = ["lateral_speed_p75", "reversal_recovery"]


def preprocessing(numeric, categorical):
    return ColumnTransformer([
        ("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        ("categorical", make_pipeline(SimpleImputer(strategy="most_frequent"),
                                      OneHotEncoder(handle_unknown="ignore")), categorical),
    ])


def player_model(numeric):
    return make_pipeline(preprocessing(numeric, ["nfl_position"]), Ridge(alpha=10, solver="lsqr"))


def load_data():
    players = pd.read_csv(DATA / "players.csv")
    table = players.merge(pd.read_csv(DATA / "combine_results.csv"), on=["nfl_id", "draft_year"], validate="one_to_one")
    table = table.merge(pd.read_csv(OUT / "mirror_player_features.csv"), on="nfl_id", validate="one_to_one")
    plays = pd.read_csv(OUT / "ol_rookie_plays.csv", low_memory=False)
    plays = plays.loc[plays.lined_up_position.isin(["T", "G", "C"]) & plays.pressure_allowed.notna() &
                      plays.pass_rushers_encountered.eq(1)].copy()
    plays["pressure"] = plays.pressure_allowed.map({True: 1, False: 0, "True": 1, "False": 0})
    plays["pressure_prob"] = pd.to_numeric(plays.peak_pressure_probability_allowed, errors="coerce")
    assert plays.pressure.notna().all() and plays.pressure_prob.notna().all()
    plays["duration_sq"] = plays.dropback_duration ** 2
    count = plays.groupby("nfl_id").size()
    table = table.loc[table.nfl_id.isin(count[count >= 100].index)].copy()
    table["opportunities"] = table.nfl_id.map(count)
    plays = plays.loc[plays.nfl_id.isin(table.nfl_id)].copy()
    return table, plays


def main():
    table, plays = load_data()
    assert not plays.duplicated(["game_id", "play_id", "nfl_id"]).any()
    table.to_csv(OUT / "mirror_model_players.csv", index=False)
    scores, predictions, context_checks = [], [], []
    categorical = ["lined_up_position", "offense_formation"]
    variants = [
        "primary_with_duration",
        "sensitivity_without_duration",
        "continuous_pressure_probability",
    ]
    for variant in variants:
        numeric = ["down", "yards_to_go"]
        if variant != "sensitivity_without_duration":
            numeric += ["dropback_duration", "duration_sq"]
        for year in (2024, 2025):
            train_p = plays.loc[plays.draft_year.lt(year)].copy()
            test_p = plays.loc[plays.draft_year.eq(year)].copy()
            if variant == "continuous_pressure_probability":
                context = make_pipeline(preprocessing(numeric, categorical), Ridge(alpha=10, solver="lsqr"))
                context.fit(train_p[numeric + categorical], 100 * train_p.pressure_prob)
                for frame in (train_p, test_p):
                    frame["expected"] = np.clip(context.predict(frame[numeric + categorical]) / 100.0, 0.0, 1.0)
                    frame["target"] = 100 * (frame.pressure_prob - frame.expected)
                obs_col = "pressure_prob"
                brier_val = float(np.mean((test_p.pressure_prob - test_p.expected) ** 2))
            else:
                context = make_pipeline(preprocessing(numeric, categorical), LogisticRegression(C=1, max_iter=2000))
                context.fit(train_p[numeric + categorical], train_p.pressure)
                for frame in (train_p, test_p):
                    frame["expected"] = context.predict_proba(frame[numeric + categorical])[:, 1]
                    frame["target"] = 100 * (frame.pressure - frame.expected)
                obs_col = "pressure"
                brier_val = float(brier_score_loss(test_p.pressure, test_p.expected))
            train = table.merge(train_p.groupby("nfl_id").target.mean(), on="nfl_id")
            test = table.merge(test_p.groupby("nfl_id").target.mean(), on="nfl_id")
            assert set(train.nfl_id).isdisjoint(test.nfl_id)
            assert train.draft_year.max() < test.draft_year.min()
            assert len(train) >= 12 and len(test) >= 5
            pred = test[["nfl_id", "display_name", "draft_year", "nfl_position", "opportunities", "target"]].copy()
            pred["variant"] = variant
            pred["mean_reference"] = train.target.mean()
            models = [("classic", CLASSIC), ("whole_drill", CLASSIC + ["whole_drill_speed"]),
                      ("mirror_phase", CLASSIC + MIRROR)]
            for name, columns in models:
                model = player_model(columns)
                model.fit(train[columns + ["nfl_position"]], train.target)
                pred[name] = model.predict(test[columns + ["nfl_position"]])
            predictions.append(pred)
            for name in ["mean_reference", "classic", "whole_drill", "mirror_phase"]:
                error = pred.target - pred[name]
                delta = (pred.target - pred["classic"]).abs().to_numpy() - error.abs().to_numpy()
                rng = np.random.default_rng(2027)
                bootstrap = rng.choice(delta, size=(5000, len(delta)), replace=True).mean(axis=1)
                scores.append({"variant": variant, "test_year": year, "model": name,
                               "train_players": len(train), "test_players": len(test),
                               "train_plays": len(train_p), "test_plays": len(test_p),
                               "mae_pp": float(error.abs().mean()), "rmse_pp": float(np.sqrt((error ** 2).mean())),
                               "improvement_over_classic_pp": float(delta.mean()),
                               "ci_low": float(np.quantile(bootstrap, .025)), "ci_high": float(np.quantile(bootstrap, .975))})
            context_checks.append({"variant": variant, "test_year": year,
                                   "observed_rate": float(test_p[obs_col].mean()),
                                   "predicted_rate": float(test_p.expected.mean()),
                                   "brier": brier_val})
    result = pd.DataFrame(scores)
    result.to_csv(OUT / "mirror_forward_year_scores.csv", index=False)
    pd.concat(predictions, ignore_index=True).to_csv(OUT / "mirror_predictions.csv", index=False)
    pd.DataFrame(context_checks).to_csv(OUT / "mirror_context_checks.csv", index=False)
    game_keys = plays[["nfl_id", "game_id"]].drop_duplicates().sort_values(["nfl_id", "game_id"])
    game_keys["half"] = game_keys.groupby("nfl_id").cumcount() % 2
    split = plays.merge(game_keys, on=["nfl_id", "game_id"], validate="many_to_one")
    halves = split.groupby(["nfl_id", "half"]).pressure.agg(["mean", "size"]).unstack()
    half_eligible = halves["size"].ge(20).all(axis=1)
    stability = halves.loc[half_eligible, "mean"]
    halves_prob = split.groupby(["nfl_id", "half"]).pressure_prob.agg(["mean", "size"]).unstack()
    stability_prob = halves_prob.loc[half_eligible, "mean"]
    summary = {"status": "exploratory research; not a finalized competition entry",
               "players": len(table), "plays": len(plays), "pressures": int(plays.pressure.sum()),
               "players_by_year": table.groupby("draft_year").size().to_dict(),
               "classic_missing": table[CLASSIC].isna().sum().to_dict(),
               "raw_game_split_half_players": len(stability),
               "raw_game_split_half_correlation": float(stability[0].corr(stability[1])),
               "continuous_pressure_prob_split_half_correlation": float(stability_prob[0].corr(stability_prob[1])),
               "scores": scores, "limitations": [
                   "One mirror attempt per player; boundary robustness is not repeatability.",
                   "Small cohorts and multiple research questions across the project.",
                   "Pressure labels and dropback duration are supplied game outcomes; context adjustment is not causal.",
                   "Conditional bootstrap excludes training uncertainty.",
                   "Mirror phase includes the transition into exit until the geometric threshold is crossed.",
                   "The drill is coach-directed; faster movement can reflect different instructions."]}
    (OUT / "mirror_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.8))
    panel_specs = [
        ("primary_with_duration", "A. Primary: Binary Pressure\n(Duration-Adjusted Residual, pp)"),
        ("sensitivity_without_duration", "B. Sensitivity: Binary Pressure\n(No Dropback-Duration Adjustment, pp)"),
        ("continuous_pressure_probability", "C. Continuous Sensitivity:\nPeak Pressure Probability (pp)"),
    ]
    palette = [("classic", "Classic Combine", "#4a6fe3"),
               ("whole_drill", "+ Whole-Drill Speed (Exit)", "#e07a2b"),
               ("mirror_phase", "+ Mirror-Phase Features", "#176b87")]
    for ax, (variant, title) in zip(axes, panel_specs):
        selection = result.loc[result.variant.eq(variant)]
        for i, (name, label, color) in enumerate(palette):
            rows = selection.loc[selection.model.eq(name)].sort_values("test_year")
            xpos = np.arange(2) + (i - 1) * .25
            bars = ax.bar(xpos, rows.mae_pp, width=.24, label=label, color=color, edgecolor="white", linewidth=0.7)
            for bar, val in zip(bars, rows.mae_pp):
                ax.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f"{val:.2f}",
                        ha="center", va="bottom", fontsize=7.5, color="#1f2d3d", fontweight="semibold")
        mean_ref = selection.loc[selection.model.eq("mean_reference")].sort_values("test_year")
        for idx_yr, ref_val in enumerate(mean_ref.mae_pp):
            ax.hlines(ref_val, idx_yr - 0.38, idx_yr + 0.38, colors="#7f8c8d", linestyles="--", linewidth=1.2,
                      label="Training Mean Reference" if idx_yr == 0 else None)
        ax.set_xticks([0, 1], ["2024 Rookie Class\n(Train: 17, Test: 16)", "2025 Rookie Class\n(Train: 33, Test: 17)"])
        ax.set_title(title, fontsize=10.5, fontweight="bold", pad=8)
        ax.set_ylim(0, max(selection.mae_pp) * 1.22)
        ax.grid(axis="y", linestyle=":", alpha=0.45)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    axes[0].set_ylabel("Out-of-Time Player MAE (percentage points; lower is better)", fontsize=9.5)
    axes[0].legend(fontsize=8, loc="upper left", frameon=True)
    fig.suptitle("Forward-Year One-on-One Pass-Protection Prediction Across Outcome Specifications (50 OL, 20,853 Snaps)",
                 fontsize=12.5, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(OUT / "mirror_model_comparison.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(result.to_string(index=False))
    print(json.dumps({key: val for key, val in summary.items() if key != "scores"}, indent=2))


main()

In [ ]:
display(Image(filename=str(WORK_DIR / 'reports/trenches/mirror_model_comparison.png')))
display(pd.read_csv(WORK_DIR / 'reports/trenches/mirror_forward_year_scores.csv'))


## 4. Intermediate Construct Validation: Early-Game Lateral Speed (0–1.5 s)

Specified after observing the primary pressure results, this check tests whether
Combine mirror lateral speed transfers directly to on-field lateral speed during the
first `1.5 seconds` after snap (`20,110` valid plays across the `50` linemen, `>= 50`
plays per player), using the exact same Savitzky–Golay differentiation (`yd/s`) and
training-only context adjustment (`lined_up_position`, `offense_formation`, `down`,
`yards_to_go`, `snap_speed`).


In [ ]:
"""Fixed 1.5-second early pass-protection movement for the mirror study cohort."""
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

ROOT = WORK_DIR
DATA = DATA_DIR
OUT = ROOT / "reports/trenches"
KEYS = ["game_id", "play_id", "nfl_id"]
END_EVENTS = ["pass_forward", "tackle", "out_of_bounds", "qb_sack", "qb_strip_sack",
              "touchdown", "pass_outcome_incomplete", "fumble"]


def main():
    table = pd.read_csv(OUT / "mirror_model_players.csv")
    metadata = pd.read_csv(OUT / "ol_rookie_plays.csv", low_memory=False)
    metadata = metadata.loc[metadata.nfl_id.isin(table.nfl_id) & metadata.lined_up_position.isin(["T", "G", "C"]) &
                            metadata.pressure_allowed.notna() & metadata.pass_rushers_encountered.eq(1)].copy()
    assert not metadata.duplicated(KEYS).any()
    outputs, exclusions, audits = [], [], []
    for year in (2023, 2024, 2025):
        year_meta = metadata.loc[metadata.draft_year.eq(year)]
        year_ids = set(year_meta.nfl_id)
        year_games = set(year_meta.game_id)
        wanted = pd.MultiIndex.from_frame(year_meta[KEYS])
        frames = []
        for chunk in pd.read_csv(DATA / f"game_tracking_{year}.csv", chunksize=500000,
                                 usecols=KEYS + ["time", "x", "y", "s", "event"]):
            chunk = chunk.loc[chunk.nfl_id.isin(year_ids) & chunk.game_id.isin(year_games)]
            if len(chunk):
                chunk = chunk.loc[pd.MultiIndex.from_frame(chunk[KEYS]).isin(wanted)]
                if len(chunk):
                    frames.append(chunk)
        tracking = pd.concat(frames, ignore_index=True)
        tracking["time"] = pd.to_datetime(tracking.time)
        tracked = pd.MultiIndex.from_frame(tracking[KEYS].drop_duplicates())
        missing = year_meta.loc[~pd.MultiIndex.from_frame(year_meta[KEYS]).isin(tracked)]
        counts = Counter(missing_tracking=len(missing))
        exclusions.extend([{**row, "reason": "missing_tracking"} for row in missing[KEYS].to_dict("records")])
        for key, g in tracking.groupby(KEYS, sort=False):
            g = g.sort_values("time")
            snap = g.loc[g.event.eq("ball_snap"), "time"]
            reason = None
            if len(snap) != 1:
                reason = "missing_or_multiple_snap"
            elif g.time.duplicated().any():
                reason = "duplicate_time"
            if not reason:
                start, end = snap.iloc[0], snap.iloc[0] + pd.Timedelta(seconds=1.5)
                window = g.loc[g.time.ge(start) & g.time.le(end)]
                if (g.event.isin(END_EVENTS) & g.time.ge(start) & g.time.lt(end)).any():
                    reason = "early_throw_or_end"
                elif len(window) < 16 or window.time.iloc[-1] != end:
                    reason = "incomplete_window"
                elif (window.time.diff().dt.total_seconds().dropna() > .21).any():
                    reason = "frame_gap"
                elif not np.isfinite(window[["x", "y", "s"]].to_numpy()).all():
                    reason = "nonfinite_values"
            if reason:
                exclusions.append({**dict(zip(KEYS, key)), "reason": reason})
                counts[reason] += 1
                continue
            time = (window.time - start).dt.total_seconds().to_numpy()
            y = savgol_filter(window.y.to_numpy(), 5, 2)
            vy = np.gradient(y, time)
            outputs.append({**dict(zip(KEYS, key)), "game_lateral_speed_p75": float(np.quantile(np.abs(vy), .75)),
                            "snap_speed": float(window.s.iloc[0]), "frames": len(window)})
            counts["accepted"] += 1
        audits.append({"year": year, "metadata_plays": len(year_meta), "counts": dict(counts)})
        assert sum(counts.values()) == len(year_meta)
        print(f"{year}: {dict(counts)}", flush=True)
    result = pd.DataFrame(outputs).merge(metadata, on=KEYS, validate="one_to_one")
    result.to_csv(OUT / "ol_early_movement.csv", index=False)
    pd.DataFrame(exclusions).to_csv(OUT / "ol_early_movement_exclusions.csv", index=False)
    audit = {"years": audits, "accepted_plays": len(result), "accepted_players": result.nfl_id.nunique()}
    (OUT / "ol_movement_extraction_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print(json.dumps(audit, indent=2))


main()

In [ ]:
"""Follow-up construct validation: mirror lateral motion to early game motion."""
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline


ROOT = WORK_DIR
OUT = ROOT / "reports/trenches"


def main():
    table = pd.read_csv(OUT / "mirror_model_players.csv")
    plays = pd.read_csv(OUT / "ol_early_movement.csv", low_memory=False)
    counts = plays.groupby("nfl_id").size()
    table = table.loc[table.nfl_id.isin(counts[counts >= 50].index)].copy()
    plays = plays.loc[plays.nfl_id.isin(table.nfl_id)].copy()
    table["movement_plays"] = table.nfl_id.map(counts)
    table.to_csv(OUT / "movement_model_players.csv", index=False)
    scores, predictions = [], []
    numeric = ["down", "yards_to_go", "snap_speed"]
    categorical = ["lined_up_position", "offense_formation"]
    for year in (2024, 2025):
        train_p, test_p = plays.loc[plays.draft_year.lt(year)].copy(), plays.loc[plays.draft_year.eq(year)].copy()
        context = make_pipeline(preprocessing(numeric, categorical), Ridge(alpha=10, solver="lsqr"))
        context.fit(train_p[numeric + categorical], train_p.game_lateral_speed_p75)
        for frame in (train_p, test_p):
            frame["target"] = frame.game_lateral_speed_p75 - context.predict(frame[numeric + categorical])
        train = table.merge(train_p.groupby("nfl_id").target.mean(), on="nfl_id")
        test = table.merge(test_p.groupby("nfl_id").target.mean(), on="nfl_id")
        assert set(train.nfl_id).isdisjoint(test.nfl_id)
        assert train.draft_year.max() < test.draft_year.min()
        pred = test[["nfl_id", "display_name", "draft_year", "nfl_position", "movement_plays", "target"]].copy()
        pred["mean_reference"] = train.target.mean()
        for name, columns in [("classic", CLASSIC), ("whole_drill", CLASSIC + ["whole_drill_speed"]),
                              ("mirror_lateral", CLASSIC + ["lateral_speed_p75"])]:
            model = player_model(columns).fit(train[columns + ["nfl_position"]], train.target)
            pred[name] = model.predict(test[columns + ["nfl_position"]])
        predictions.append(pred)
        for name in ["mean_reference", "classic", "whole_drill", "mirror_lateral"]:
            error = pred.target - pred[name]
            delta = (pred.target - pred["classic"]).abs().to_numpy() - error.abs().to_numpy()
            rng = np.random.default_rng(2027)
            boot = rng.choice(delta, size=(5000, len(delta)), replace=True).mean(axis=1)
            scores.append({"test_year": year, "model": name, "train_players": len(train), "test_players": len(test),
                           "train_plays": len(train_p), "test_plays": len(test_p),
                           "mae": float(error.abs().mean()), "rmse": float(np.sqrt(np.mean(error ** 2))),
                           "improvement_over_classic": float(delta.mean()),
                           "ci_low": float(np.quantile(boot, .025)), "ci_high": float(np.quantile(boot, .975))})
    scores = pd.DataFrame(scores)
    scores.to_csv(OUT / "movement_forward_year_scores.csv", index=False)
    pd.concat(predictions, ignore_index=True).to_csv(OUT / "movement_predictions.csv", index=False)
    raw = table.merge(plays.groupby("nfl_id").game_lateral_speed_p75.mean(), on="nfl_id")
    raw.to_csv(OUT / "movement_player_means.csv", index=False)
    cols = ["lateral_speed_p75", "game_lateral_speed_p75", "whole_drill_speed"]
    demeaned = raw[cols] - raw.groupby(["draft_year", "nfl_position"])[cols].transform("mean")
    correlations = []
    for name, frame in [("pooled", raw)] + [(str(y), g) for y, g in raw.groupby("draft_year")] + [
            ("within_year_position", demeaned)]:
        correlations.append({"group": name, "players": len(frame),
                             "mirror_vs_game": float(frame.lateral_speed_p75.corr(frame.game_lateral_speed_p75)),
                             "whole_drill_vs_game": float(frame.whole_drill_speed.corr(frame.game_lateral_speed_p75))})
    pd.DataFrame(correlations).to_csv(OUT / "movement_correlations.csv", index=False)
    game_order = plays[["nfl_id", "game_id"]].drop_duplicates().sort_values(["nfl_id", "game_id"])
    game_order["half"] = game_order.groupby("nfl_id").cumcount() % 2
    split = plays.merge(game_order, on=["nfl_id", "game_id"], validate="many_to_one")
    halves = split.groupby(["nfl_id", "half"]).game_lateral_speed_p75.agg(["mean", "size"]).unstack()
    half_eligible = halves["size"].ge(20).all(axis=1)
    means = halves.loc[half_eligible, "mean"]
    summary = {"status": "follow-up exploratory construct validation; pressure result remains the primary outcome",
               "players": len(table), "plays": len(plays), "game_split_half_players": len(means),
               "game_split_half_correlation": float(means[0].corr(means[1])),
               "correlations": correlations, "scores": scores.to_dict("records"),
               "limitations": ["Early lateral movement reflects assignment and opponent response, not necessarily superior skill.",
                               "One Combine mirror attempt per player; coach instruction is unobserved.",
                               "This follow-up was specified after the pressure outcome analysis.",
                               "Within-year/position demeaning is descriptive, not independent validation."]}
    (OUT / "movement_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6))
    pos_colors = {"T": "#176b87", "G": "#e07a2b", "C": "#2e8b57"}
    for position, group in raw.groupby("nfl_position"):
        axes[0].scatter(group.lateral_speed_p75, group.game_lateral_speed_p75,
                        label=position, color=pos_colors.get(position, "#334455"), s=42, alpha=.85, edgecolor="white", linewidth=0.5)
    axes[0].set(xlabel="Combine Mirror Lateral Speed, 75th Percentile (yd/s)",
                ylabel="Mean Rookie Early-Game Lateral Speed, 0–1.5 s (yd/s)",
                title="A. Same Axis, Different Tasks (Pooled r = +0.053, Demeaned r = +0.120)")
    axes[0].grid(linestyle=":", alpha=0.45)
    axes[0].spines["top"].set_visible(False)
    axes[0].spines["right"].set_visible(False)
    axes[0].legend(title="Position", frameon=True, fontsize=8.5)
    palette = [("classic", "Classic Combine", "#4a6fe3"),
               ("whole_drill", "+ Whole-Drill Speed", "#e07a2b"),
               ("mirror_lateral", "+ Mirror Lateral Speed", "#176b87")]
    for i, (name, label, color) in enumerate(palette):
        rows = scores.loc[scores.model.eq(name)].sort_values("test_year")
        xpos = np.arange(2) + (i - 1) * .25
        bars = axes[1].bar(xpos, rows.mae, width=.24, label=label, color=color, edgecolor="white", linewidth=0.7)
        for bar, val in zip(bars, rows.mae):
            axes[1].text(bar.get_x() + bar.get_width() / 2, val + 0.002, f"{val:.3f}",
                         ha="center", va="bottom", fontsize=8, color="#1f2d3d", fontweight="semibold")
    axes[1].set(xticks=[0, 1], xticklabels=["2024 Rookie Class\n(Train: 17, Test: 16)", "2025 Rookie Class\n(Train: 33, Test: 17)"],
                ylabel="Out-of-Time Player MAE (yd/s; lower is better)",
                title="B. Forward-Year Early-Game Lateral Speed Prediction")
    axes[1].set_ylim(0, max(scores.mae) * 1.22)
    axes[1].grid(axis="y", linestyle=":", alpha=0.45)
    axes[1].spines["top"].set_visible(False)
    axes[1].spines["right"].set_visible(False)
    axes[1].legend(fontsize=8.5, frameon=True)
    fig.tight_layout()
    fig.savefig(OUT / "movement_bridge_results.png", dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(json.dumps(summary, indent=2))


main()

In [ ]:
display(Image(filename=str(WORK_DIR / 'reports/trenches/movement_bridge_results.png')))
display(pd.read_csv(WORK_DIR / 'reports/trenches/movement_forward_year_scores.csv'))


## 5. Diagnostic Visualization and Bilingual Interactive Explorer

Within-class percentile comparisons (`Spearman r = +0.005`) demonstrate that
exit-contaminated whole-drill speed and isolated mirror lateral speed produce nearly
orthogonal rankings. This section generates `mirror_phase_story.png`,
`mirror_ranking_shift.png`, and the standalone bilingual (`TR` / `EN`) interactive
explorer `mirror_explorer.html` covering all `113` accepted offensive linemen.


In [ ]:
"""Measurement diagnostics and an offline per-player phase explorer."""
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = WORK_DIR
DATA = DATA_DIR
OUT = ROOT / "reports/trenches"


def main():
    attempts = pd.read_csv(OUT / "mirror_attempt_features.csv")
    players = pd.read_csv(DATA / "players.csv")
    ranked = attempts.merge(players[["nfl_id", "display_name", "nfl_position"]], on="nfl_id", validate="one_to_one")
    for feature in ["lateral_speed_p75", "whole_drill_speed"]:
        ranked[feature + "_percentile"] = ranked.groupby("draft_year")[feature].rank(pct=True) * 100
    ranked["rank_difference_pp"] = ranked.lateral_speed_p75_percentile - ranked.whole_drill_speed_percentile
    ranked.to_csv(OUT / "mirror_measurement_rank_comparison.csv", index=False)
    by_year = []
    for year, group in ranked.groupby("draft_year"):
        lateral_top = group.lateral_speed_p75_percentile.ge(75)
        full_top = group.whole_drill_speed_percentile.ge(75)
        by_year.append({"year": int(year), "players": len(group),
                        "rank_correlation": float(group.lateral_speed_p75.corr(group.whole_drill_speed, method="spearman")),
                        "median_absolute_percentile_change": float(group.rank_difference_pp.abs().median()),
                        "mirror_top_quartile": int(lateral_top.sum()), "whole_drill_top_quartile": int(full_top.sum()),
                        "top_quartile_overlap": int((lateral_top & full_top).sum()),
                        "median_reversals": float(group.reversals.median())})
    suite_path = OUT / "ol_drill_suite_audit.csv"
    suite_records = pd.read_csv(suite_path).to_dict("records") if suite_path.exists() else []
    report = {"players": len(ranked), "peak_in_exit_fraction": float(ranked.peak_in_exit.mean()),
              "pooled_rank_correlation": float(ranked.lateral_speed_p75.corr(ranked.whole_drill_speed, method="spearman")),
              "by_year": by_year,
              "ol_drill_suite": suite_records,
              "interpretation": "These are rankings of distinct observed movement summaries, not rankings of football talent."}
    (OUT / "mirror_measurement_summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    tracking = pd.read_csv(DATA / "combine_tracking.csv")
    tracking = tracking.loc[tracking.entity_type.eq("PLAYER") &
                            tracking.drill_name.isin(["PASS_PRO-MIRROR_DRILL", "PASS_PRO_MIRROR_DRILL"])].copy()
    tracking["time"] = pd.to_datetime(tracking.time)
    traces = []
    examples = {}
    for row in ranked.itertuples():
        g = tracking.loc[tracking.nfl_id.eq(row.nfl_id) & tracking.event_id.eq(row.event_id) & tracking.attempt.eq(row.attempt)]
        record, reason, detail = measure(g)
        assert reason is None
        g, end, turns, vy = detail
        t = (g.time - g.time.iloc[0]).dt.total_seconds().to_numpy()
        x0_anchor = float(np.median(g.x.to_numpy()[:5]))
        vy_full = np.zeros(len(g), dtype=float)
        vy_full[:end] = np.abs(vy)
        traces.append({
            "id": int(row.nfl_id),
            "name": row.display_name,
            "year": int(row.draft_year),
            "position": row.nfl_position,
            "time": t.round(3).tolist(),
            "x": g.x.round(4).tolist(),
            "y": g.y.round(4).tolist(),
            "s": g.s.round(4).tolist(),
            "vy": vy_full.round(4).tolist(),
            "end": int(end),
            "x_exit": round(x0_anchor + 1.5, 4),
            "turns": [int(i) for i, _ in turns],
            "lateral_speed": round(float(row.lateral_speed_p75), 2),
            "whole_speed": round(float(row.whole_drill_speed), 2),
            "recovery": round(float(row.reversal_recovery), 2),
            "duration": round(float(row.mirror_duration), 1),
            "lateral_percentile": round(float(row.lateral_speed_p75_percentile), 1),
            "whole_percentile": round(float(row.whole_drill_speed_percentile), 1),
            "diff_pp": round(float(row.rank_difference_pp), 1),
        })
        if row.draft_year not in examples:
            examples[row.draft_year] = (g, end, turns, vy, t, row.display_name, row.nfl_position)

    # Figure 1: mirror_phase_story.png
    fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.6))
    for col, (year, (g, end, turns, vy, t, name, pos)) in enumerate(sorted(examples.items())):
        ax = axes[0, col]
        x0_anchor = float(np.median(g.x.to_numpy()[:5]))
        ax.plot(g.x.iloc[:end], g.y.iloc[:end], color="#176b87", linewidth=2.6, label="Mirror phase (lateral slide)")
        ax.plot(g.x.iloc[end - 1:], g.y.iloc[end - 1:], color="#e07a2b", linewidth=2.6, label="Exit sprint phase")
        ax.axvline(x0_anchor + 1.5, color="#6c7a89", linestyle="--", linewidth=1.2, label="Exit threshold (+1.5 yd)")
        ax.scatter(g.x.iloc[0], g.y.iloc[0], color="#1f2d3d", marker="X", s=55, zorder=5, label="Drill start")
        if turns:
            t_idx = [i for i, _ in turns]
            ax.scatter(g.x.iloc[t_idx], g.y.iloc[t_idx], color="#b43b45", edgecolor="white", linewidth=0.6,
                       s=46, zorder=6, label=f"Detected reversals (n={len(t_idx)})")
        ax.set_title(f"{year} Class: {name} ({pos})", fontsize=11, fontweight="bold")
        ax.set_aspect("equal", adjustable="datalim")
        ax.set(xlabel="Field x coordinate (yards)", ylabel="Field y coordinate (yards)")
        ax.grid(linestyle=":", alpha=0.45)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if col == 0:
            ax.legend(fontsize=7.8, loc="best", frameon=True)

        ax = axes[1, col]
        smooth = g.s.rolling(3, center=True, min_periods=1).median()
        ax.plot(t, smooth, color="#2c3e50", linewidth=2.1, label="Total speed s(t) (yd/s)")
        ax.plot(t[:end], np.abs(vy), color="#176b87", linestyle="--", linewidth=1.6, alpha=0.9,
                label="Mirror lateral speed |vy(t)|")
        ax.axvspan(t[end], t[-1], color="#e07a2b", alpha=.22, label="Exit sprint window")
        idx = int(np.argmax(smooth.to_numpy()))
        ax.scatter(t[idx], smooth.iloc[idx], color="#b43b45", edgecolor="white", linewidth=0.7,
                   s=58, zorder=6, label=f"Peak speed: {smooth.iloc[idx]:.1f} yd/s")
        ax.set(xlabel="Time from drill start (seconds)", ylabel="Speed (yd/s)")
        ax.set_ylim(0, max(smooth) * 1.18)
        ax.grid(linestyle=":", alpha=0.45)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        if col == 0:
            ax.legend(fontsize=7.8, loc="upper left", frameon=True)

    fig.suptitle("In All 113 Accepted Mirror Recordings, Peak Speed Occurs in the Finishing Exit Sprint",
                 fontsize=13.5, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(OUT / "mirror_phase_story.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    # Figure 2: mirror_ranking_shift.png (2-panel: Panel A Rank Shift + Panel B 3-Drill OL Suite Contamination)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 5.4), gridspec_kw={"width_ratios": [1.15, 1.0]})
    year_colors = {2023: "#176b87", 2024: "#e07a2b", 2025: "#2e8b57"}
    ax1.axvspan(75, 102, ymin=0, ymax=1, color="#e07a2b", alpha=0.07)
    ax1.axhspan(75, 102, xmin=0, xmax=1, color="#176b87", alpha=0.07)
    ax1.axvline(75, color="#e07a2b", linestyle=":", linewidth=1.0, alpha=0.7)
    ax1.axhline(75, color="#176b87", linestyle=":", linewidth=1.0, alpha=0.7)
    for year, group in ranked.groupby("draft_year"):
        ax1.scatter(group.whole_drill_speed_percentile, group.lateral_speed_p75_percentile,
                    label=f"{year} (n={len(group)})", color=year_colors.get(year, "#334455"),
                    s=44, alpha=.85, edgecolor="white", linewidth=0.5)
    ax1.plot([0, 100], [0, 100], "--", color="#7f8c8d", linewidth=1.1, label="Identity (no rank change)")
    callouts = [
        ("Donovan Jackson", (14, 93)),
        ("Dominick Puni", (18, 82)),
        ("Darnell Wright", (32, 9)),
    ]
    for cname, text_xy in callouts:
        hit = ranked.loc[ranked.display_name.eq(cname)]
        if not hit.empty:
            r0 = hit.iloc[0]
            ax1.annotate(
                f"{r0.display_name} ({r0.rank_difference_pp:+.0f} pp)",
                xy=(r0.whole_drill_speed_percentile, r0.lateral_speed_p75_percentile),
                xytext=text_xy,
                fontsize=8, fontweight="semibold", color="#1f2d3d",
                arrowprops=dict(arrowstyle="->", color="#526477", lw=0.9),
            )
    ax1.set_xlim(0, 103)
    ax1.set_ylim(0, 103)
    ax1.set(xlabel="Whole-Drill Speed Percentile Within Class (95th pct, Exit-Contaminated)",
            ylabel="Mirror-Phase Lateral Speed Percentile Within Class (75th pct)",
            title="A. Mirror Drill Re-Ranking Within Class (Spearman r = +0.005)")
    ax1.grid(linestyle=":", alpha=0.45)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    ax1.legend(title="Draft Class", fontsize=8, loc="lower right", frameon=True)

    if suite_records:
        suite_df = pd.DataFrame(suite_records)
    else:
        suite_df = pd.DataFrame([
            {"drill": "Pass-Pro Mirror Drill", "accepted_attempts": 113, "peak_in_exit_fraction": 1.0, "phase_vs_whole_spearman": 0.005},
            {"drill": "5-Yard Wave Drill (Slide & Shuffle)", "accepted_attempts": 118, "peak_in_exit_fraction": 1.0, "phase_vs_whole_spearman": -0.022},
            {"drill": "Pass Rush Drop (Kick-Slide)", "accepted_attempts": 239, "peak_in_exit_fraction": 1.0, "phase_vs_whole_spearman": 0.147},
        ])
    y_pos = np.arange(len(suite_df))
    exit_pct = suite_df.peak_in_exit_fraction * 100
    corr_pct = suite_df.phase_vs_whole_spearman.abs() * 100
    short_labels = [
        f"Pass-Pro Mirror\n(n = {suite_df.iloc[0].accepted_attempts} reps)",
        f"5-Yard Wave Shuffle\n(n = {suite_df.iloc[1].accepted_attempts} reps)",
        f"Pass Rush Drop\n(n = {suite_df.iloc[2].accepted_attempts} reps, 2/player)",
    ]
    bars1 = ax2.barh(y_pos - 0.17, exit_pct, height=0.32, color="#e07a2b", edgecolor="white",
                     label="Peak speed occurs in exit sprint (%)")
    bars2 = ax2.barh(y_pos + 0.17, corr_pct, height=0.32, color="#176b87", edgecolor="white",
                     label="|Spearman r| between lateral phase & whole-drill speed (%)")
    for bar, val in zip(bars1, exit_pct):
        ax2.text(val + 1.5, bar.get_y() + bar.get_height() / 2, f"{val:.0f}%",
                 va="center", ha="left", fontsize=8.5, fontweight="bold", color="#b85d19")
    for bar, orig_r in zip(bars2, suite_df.phase_vs_whole_spearman):
        ax2.text(abs(orig_r) * 100 + 1.5, bar.get_y() + bar.get_height() / 2, f"r = {orig_r:+.3f}",
                 va="center", ha="left", fontsize=8.5, fontweight="bold", color="#176b87")
    ax2.set_yticks(y_pos, short_labels, fontsize=9)
    ax2.set_ylim(-0.55, 3.15)
    ax2.invert_yaxis()
    ax2.set_xlim(0, 118)
    ax2.set_xlabel("Percentage (%) / Magnitude of Rank Correlation", fontsize=9.5)
    ax2.set_title("B. Universal Exit Contamination Across All 3 OL Drills (470/470 Reps)", fontsize=10.5, fontweight="bold")
    ax2.grid(axis="x", linestyle=":", alpha=0.45)
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    ax2.legend(fontsize=7.8, loc="lower right", frameon=True)
    ax2.text(2, 2.72,
             "2-Rep Repeatability (Pass Rush Drop, n=118 OL):\n• Exit sprint peak speed: r = +0.510\n• Kick-slide lateral speed: r = +0.104 (attenuates transfer)",
             fontsize=7.6, color="#1f2d3d", bbox=dict(boxstyle="round,pad=0.38", facecolor="#f4f6f9", edgecolor="#cbd5e1"))

    fig.suptitle("Phase Contamination Distorts Rankings Across the Entire Offensive Line Combine Drill Suite",
                 fontsize=13, fontweight="bold", y=1.01)
    fig.tight_layout()
    fig.savefig(OUT / "mirror_ranking_shift.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

    data = json.dumps(traces, ensure_ascii=False).replace("</", "<\\/")
    page = '''<!doctype html><html lang="tr"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>NFL Big Data Bowl 2027 — OL Combine Phase Explorer</title>
<style>
:root{--bg:#f3f6f9;--card:#ffffff;--ink:#172638;--muted:#526477;--blue:#176b87;--orange:#e07a2b;--red:#b43b45;--border:#dce4ed}
*{box-sizing:border-box}
body{font:15px/1.6 system-ui,-apple-system,Segoe UI,Roboto,sans-serif;background:var(--bg);color:var(--ink);margin:0}
main{max-width:1160px;margin:auto;padding:24px 20px}
header{display:flex;justify-content:space-between;align-items:flex-start;gap:16px;flex-wrap:wrap;margin-bottom:16px}
h1{font-size:24px;margin:0 0 6px 0;line-height:1.25}
h2{font-size:18px;margin:0 0 10px 0}
h3{font-size:15px;margin:0 0 8px 0}
p{margin:6px 0}
section{background:var(--card);border:1px solid var(--border);border-radius:12px;padding:20px;margin:16px 0;box-shadow:0 1px 3px rgba(0,0,0,.03)}
.controls{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;margin-bottom:16px}
label{display:block;font-size:12px;font-weight:600;color:var(--muted);margin-bottom:4px;text-transform:uppercase;letter-spacing:.03em}
select,button{font:inherit;padding:8px 10px;border:1px solid #cbd5e1;border-radius:8px;background:#fff;color:var(--ink);width:100%}
button.lang-btn{width:auto;padding:7px 14px;font-weight:600;cursor:pointer;background:#172638;color:#fff;border:none}
.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(175px,1fr));gap:12px;margin:14px 0}
.kpi{background:#f8fafc;border:1px solid var(--border);border-radius:10px;padding:12px}
.kpi .val{font-size:21px;font-weight:700;color:var(--ink)}
.kpi .sub{font-size:12.5px;color:var(--muted)}
.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}
@media(max-width:760px){.grid{grid-template-columns:1fr}}
svg{width:100%;height:auto;background:#fbfcfd;border:1px solid #e2e8f0;border-radius:8px}
.legend{display:flex;gap:16px;flex-wrap:wrap;font-size:13px;margin-top:10px}
.dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:5px}
table{width:100%;border-collapse:collapse;font-size:14px;margin-top:8px}
th,td{text-align:left;padding:9px 10px;border-bottom:1px solid var(--border)}
th{background:#f8fafc;color:var(--muted);font-weight:600}
.badge-pos{color:#15803d;font-weight:700}
.badge-neg{color:#b91c1c;font-weight:700}
.muted{color:var(--muted);font-size:13.5px}
</style></head><body><main>
<header>
  <div>
    <h1 id="t-title">En Hızlı Kısım Çıkış Sprinti: OL Combine Faz İnceleyici</h1>
    <p id="t-sub" class="muted">113 Offensive Lineman (T/G/C) için Pass-Pro Mirror Drill yanal ayna fazı ve çıkış sprinti ayrıştırması (Birimler: yard, yd/s, saniye).</p>
  </div>
  <button class="lang-btn" id="langBtn" onclick="toggleLang()">EN / TR</button>
</header>
<section>
  <div class="controls">
    <div><label id="l-year" for="fYear">Draft Yılı</label><select id="fYear"><option value="all">Tümü (2023–2025)</option><option value="2023">2023</option><option value="2024">2024</option><option value="2025">2025</option></select></div>
    <div><label id="l-pos" for="fPos">Pozisyon</label><select id="fPos"><option value="all">Tümü (T / G / C)</option><option value="T">Tackle (T)</option><option value="G">Guard (G)</option><option value="C">Center (C)</option></select></div>
    <div><label id="l-sort" for="fSort">Sıralama</label><select id="fSort">
      <option value="name">İsme Göre (A–Z)</option>
      <option value="lat_desc">Mirror Yanal Hız Yüzdeliği (Azalan)</option>
      <option value="whole_desc">Tüm Kayıt Hızı Yüzdeliği (Azalan)</option>
      <option value="shift_desc">En Büyük Sıralama Farkı |Δ| (Azalan)</option>
    </select></div>
    <div><label id="l-player" for="player">Oyuncu Seçimi</label><select id="player"></select></div>
  </div>
  <div class="kpis">
    <div class="kpi"><div class="sub" id="k1-t">Mirror Yanal Hız (p75)</div><div class="val" id="k1-v">-</div><div class="sub" id="k1-s">-</div></div>
    <div class="kpi"><div class="sub" id="k2-t">Tüm Kayıt Hızı (p95, Çıkış)</div><div class="val" id="k2-v">-</div><div class="sub" id="k2-s">-</div></div>
    <div class="kpi"><div class="sub" id="k3-t">Sıralama Değişimi (Yanal − Tüm)</div><div class="val" id="k3-v">-</div><div class="sub" id="k3-s">-</div></div>
    <div class="kpi"><div class="sub" id="k4-t">Dönüş Toparlanma & Süre</div><div class="val" id="k4-v">-</div><div class="sub" id="k4-s">-</div></div>
  </div>
  <div class="grid">
    <div><h3 id="c1-t">Saha İzi: x–y Koordinatları (yard)</h3><svg id="path" viewBox="0 0 500 340"></svg></div>
    <div><h3 id="c2-t">Zaman Serisi: Toplam Hız s(t) ve Yanal Hız |vy(t)| (yd/s)</h3><svg id="speed" viewBox="0 0 500 340"></svg></div>
  </div>
  <div class="legend">
    <span><i class="dot" style="background:#176b87"></i><b id="lg-1">Mavi: Mirror (Yanal Kayma) Fazı</b></span>
    <span><i class="dot" style="background:#e07a2b"></i><b id="lg-2">Turuncu: Çıkış Sprinti Fazı (+1.5 yd sonrası)</b></span>
    <span><i class="dot" style="background:#b43b45"></i><b id="lg-3">Kırmızı: Tespit Edilen Yön Değişimleri & Zirve Hız</b></span>
  </div>
  <p class="muted" id="t-note">Not: Yüzdelikler aynı draft sınıfı içindeki hareket ölçümlerinin sırasıdır; doğrudan oyuncu yeteneği sıralaması değildir.</p>
</section>
<section>
  <h2 id="s2-title">3 OL Combine Drilinde Evrensel Çıkış Sprinti Baskınlığı (470 / 470 Kayıt)</h2>
  <table>
    <thead><tr><th id="th-1">Combine Drili</th><th id="th-2">Kabul Edilen Kayıt</th><th id="th-3">Zirve Hızı Çıkış Sprintinde Olan</th><th id="th-4">Yanal Faz vs. Tüm Kayıt Hızı (Spearman r)</th><th id="th-5">2-Tekrar Güvenilirliği (Repeatability)</th></tr></thead>
    <tbody>
      <tr><td><b>Pass-Pro Mirror Drill</b></td><td>113 deneme (113 OL)</td><td><b>100.0% (113 / 113)</b></td><td>r = +0.005</td><td>Tek deneme (sınır duyarlılığı r ≥ 0.90)</td></tr>
      <tr><td><b>5-Yard Wave Drill (Slide & Shuffle)</b></td><td>118 deneme (118 OL)</td><td><b>100.0% (118 / 118)</b></td><td>r = −0.022</td><td>Tek deneme</td></tr>
      <tr><td><b>Pass Rush Drop (Kick-Slide)</b></td><td>239 deneme (120 OL)</td><td><b>100.0% (239 / 239)</b></td><td>r = +0.147</td><td>Çıkış r = +0.510 vs. Yanal r = +0.104 (118 çift)</td></tr>
    </tbody>
  </table>
  <p class="muted" id="s2-desc">50 çaylak OL oyuncusunun 20.853 bire bir pas koruma oyununda (ikili baskı ve sürekli NGS baskı olasılığı hedeflerinde) ve ilk 1.5 saniyelik maç içi yanal hız transferinde, faz ayrıştırması çıkış sprinti gürültüsünü temizlese de klasik Combine testlerine (40-yard, 10-yard split, kilo, kol boyu) istatistiksel olarak anlamlı ek tahmin gücü sağlamamaktadır (2025 tam yeniden eğitimli %95 GA: [−0.264, +0.353] yp).</p>
</section>
</main>
<script>
const rawData=__DATA__;
let lang='tr';
const i18n={
  tr:{
    title:'En Hızlı Kısım Çıkış Sprinti: OL Combine Faz İnceleyici',
    sub:'113 Offensive Lineman (T/G/C) için Pass-Pro Mirror Drill yanal ayna fazı ve çıkış sprinti ayrıştırması (Birimler: yard, yd/s, saniye).',
    lyear:'Draft Yılı',lpos:'Pozisyon',lsort:'Sıralama',lplayer:'Oyuncu Seçimi',
    allYears:'Tümü (2023–2025)',allPos:'Tümü (T / G / C)',
    sName:'İsme Göre (A–Z)',sLat:'Mirror Yanal Hız Yüzdeliği (Azalan)',sWhole:'Tüm Kayıt Hızı Yüzdeliği (Azalan)',sShift:'En Büyük Sıralama Farkı |Δ| (Azalan)',
    k1t:'Mirror Yanal Hız (p75)',k2t:'Tüm Kayıt Hızı (p95, Çıkış)',k3t:'Sıralama Değişimi (Yanal − Tüm)',k4t:'Dönüş Toparlanma & Süre',
    c1t:'Saha İzi: x–y Koordinatları (yard)',c2t:'Zaman Serisi: Toplam Hız s(t) ve Yanal Hız |vy(t)| (yd/s)',
    lg1:'Mavi: Mirror (Yanal Kayma) Fazı',lg2:'Turuncu: Çıkış Sprinti Fazı (+1.5 yd sonrası)',lg3:'Kırmızı: Tespit Edilen Yön Değişimleri & Zirve Hız',
    note:'Not: Yüzdelikler aynı draft sınıfı içindeki hareket ölçümlerinin sırasıdır; doğrudan oyuncu yeteneği sıralaması değildir.',
    s2t:'3 OL Combine Drilinde Evrensel Çıkış Sprinti Baskınlığı (470 / 470 Kayıt)',
    th1:'Combine Drili',th2:'Kabul Edilen Kayıt',th3:'Zirve Hızı Çıkış Sprintinde Olan',th4:'Yanal Faz vs. Tüm Kayıt Hızı (Spearman r)',th5:'2-Tekrar Güvenilirliği (Repeatability)',
    s2d:'50 çaylak OL oyuncusunun 20.853 bire bir pas koruma oyununda (ikili baskı ve sürekli NGS baskı olasılığı hedeflerinde) ve ilk 1.5 saniyelik maç içi yanal hız transferinde, faz ayrıştırması çıkış sprinti gürültüsünü temizlese de klasik Combine testlerine (40-yard, 10-yard split, kilo, kol boyu) istatistiksel olarak anlamlı ek tahmin gücü sağlamamaktadır (2025 tam yeniden eğitimli %95 GA: [−0.264, +0.353] yp).',
    pct:'Sınıf Yüzdeliği',turns:'dönüş',mirrorDur:'Mirror süresi',exitThresh:'Çıkış eşiği (+1.5 yd)'
  },
  en:{
    title:'The Fastest Part Is the Exit: OL Combine Phase Explorer',
    sub:'Phase-segmented Pass-Pro Mirror Drill lateral footwork vs. finishing exit sprint across 113 Offensive Linemen (Units: yards, yd/s, seconds).',
    lyear:'Draft Year',lpos:'Position',lsort:'Sort Players',lplayer:'Select Player',
    allYears:'All Classes (2023–2025)',allPos:'All Positions (T / G / C)',
    sName:'By Name (A–Z)',sLat:'Mirror Lateral Percentile (High to Low)',sWhole:'Whole-Drill Speed Percentile (High to Low)',sShift:'Largest Percentile Shift |Δ| (High to Low)',
    k1t:'Mirror Lateral Speed (p75)',k2t:'Whole-Drill Speed (p95, Exit)',k3t:'Percentile Shift (Lateral − Whole)',k4t:'Reversal Recovery & Duration',
    c1t:'Field Trajectory: x–y Coordinates (yards)',c2t:'Speed Profile: Total s(t) & Lateral |vy(t)| (yd/s)',
    lg1:'Blue: Mirror (Lateral Slide) Phase',lg2:'Orange: Exit Sprint Phase (past +1.5 yd)',lg3:'Red: Detected Reversals & Global Peak Speed',
    note:'Note: Percentiles rank observed drill movement summaries within each draft class, not overall football talent.',
    s2t:'Universal Exit-Sprint Dominance Across All 3 OL Combine Drills (470 / 470 Attempts)',
    th1:'Combine Drill',th2:'Accepted Recordings',th3:'Peak Speed in Exit Sprint',th4:'Lateral Phase vs. Whole-Drill Speed (Spearman r)',th5:'2-Attempt Repeatability',
    s2d:'Across 20,853 rookie one-on-one pass-protection snaps for 50 OL (evaluating both binary pressure allowed and continuous NGS peak pressure probability) and 0–1.5 s early-game lateral speed, phase segmentation removes exit-sprint distortion but does not yield statistically significant out-of-time gains over classic Combine metrics (2025 full-refit 95% CI: [−0.264, +0.353] pp).',
    pct:'Class Percentile',turns:'reversals',mirrorDur:'Mirror duration',exitThresh:'Exit threshold (+1.5 yd)'
  }
};
const select=document.getElementById('player'),fYear=document.getElementById('fYear'),fPos=document.getElementById('fPos'),fSort=document.getElementById('fSort');
let filtered=[];
function applyFilters(){
  const yr=fYear.value,pos=fPos.value,srt=fSort.value;
  const prevId=filtered[+select.value]?.id;
  filtered=rawData.filter(d=>(yr==='all'||String(d.year)===yr)&&(pos==='all'||d.position===pos));
  if(srt==='name')filtered.sort((a,b)=>a.year-b.year||a.name.localeCompare(b.name));
  else if(srt==='lat_desc')filtered.sort((a,b)=>b.lateral_percentile-a.lateral_percentile);
  else if(srt==='whole_desc')filtered.sort((a,b)=>b.whole_percentile-a.whole_percentile);
  else if(srt==='shift_desc')filtered.sort((a,b)=>Math.abs(b.diff_pp)-Math.abs(a.diff_pp));
  select.replaceChildren();
  let newIdx=0;
  filtered.forEach((d,i)=>{
    if(d.id===prevId)newIdx=i;
    const o=document.createElement('option');
    o.value=i;
    const sign=d.diff_pp>=0?'+':'';
    o.textContent=`${d.year} · ${d.name} (${d.position}) — Δ ${sign}${d.diff_pp} pp`;
    select.appendChild(o);
  });
  select.value=newIdx;
  draw();
}
const NS='http://www.w3.org/2000/svg';
function el(tag,attrs,parent){const n=document.createElementNS(NS,tag);for(const[k,v]of Object.entries(attrs))n.setAttribute(k,v);parent.appendChild(n);return n}
function draw(){
  const d=filtered[+select.value];
  if(!d)return;
  const t=i18n[lang];
  document.getElementById('k1-v').textContent=`${d.lateral_speed.toFixed(2)} yd/s`;
  document.getElementById('k1-s').textContent=`${t.pct}: ${d.lateral_percentile.toFixed(1)}`;
  document.getElementById('k2-v').textContent=`${d.whole_speed.toFixed(2)} yd/s`;
  document.getElementById('k2-s').textContent=`${t.pct}: ${d.whole_percentile.toFixed(1)}`;
  const sign=d.diff_pp>=0?'+':'';
  const k3v=document.getElementById('k3-v');
  k3v.textContent=`${sign}${d.diff_pp.toFixed(1)} pp`;
  k3v.className='val '+(d.diff_pp>=0?'badge-pos':'badge-neg');
  document.getElementById('k3-s').textContent=`${d.year} (${d.position})`;
  document.getElementById('k4-v').textContent=`${d.recovery.toFixed(2)}x (${d.turns.length} ${t.turns})`;
  document.getElementById('k4-s').textContent=`${t.mirrorDur}: ${d.duration.toFixed(1)} s`;

  const path=document.getElementById('path'),speed=document.getElementById('speed');
  path.replaceChildren();speed.replaceChildren();
  let xmin=Math.min(...d.x,d.x_exit),xmax=Math.max(...d.x,d.x_exit),ymin=Math.min(...d.y),ymax=Math.max(...d.y);
  const scale=Math.min(410/(xmax-xmin||1),250/(ymax-ymin||1));
  const xx=x=>48+(x-xmin)*scale,yy=y=>295-(y-ymin)*scale;
  el('line',{x1:xx(d.x_exit),y1:20,x2:xx(d.x_exit),y2:300,stroke:'#64748b','stroke-width':1.5,'stroke-dasharray':'5,4'},path);
  const bt=el('text',{x:xx(d.x_exit)+5,y:34,'font-size':11,fill:'#475569'},path);bt.textContent=t.exitThresh;
  for(let i=1;i<d.x.length;i++)el('line',{x1:xx(d.x[i-1]),y1:yy(d.y[i-1]),x2:xx(d.x[i]),y2:yy(d.y[i]),stroke:i<d.end?'#176b87':'#e07a2b','stroke-width':3},path);
  el('circle',{cx:xx(d.x[0]),cy:yy(d.y[0]),r:5,fill:'#172638'},path);
  d.turns.forEach(i=>el('circle',{cx:xx(d.x[i]),cy:yy(d.y[i]),r:4.5,fill:'#b43b45',stroke:'#fff','stroke-width':1},path));
  const xl=el('text',{x:250,y:330,'font-size':12,'text-anchor':'middle',fill:'#475569'},path);xl.textContent='x (yards)';
  const yl=el('text',{x:16,y:165,'font-size':12,'text-anchor':'middle',fill:'#475569',transform:'rotate(-90 16 165)'},path);yl.textContent='y (yards)';

  const maxS=Math.max(...d.s,4)*1.08;
  const tx=tm=>48+420*tm/d.time.at(-1),sy=s=>295-255*s/maxS;
  el('rect',{x:tx(d.time[d.end]),y:22,width:468-tx(d.time[d.end]),height:273,fill:'#fff3e0'},speed);
  for(let s=0;s<=maxS;s+=1){
    el('line',{x1:48,y1:sy(s),x2:468,y2:sy(s),stroke:'#e2e8f0','stroke-width':1},speed);
    const st=el('text',{x:40,y:sy(s)+4,'font-size':11,'text-anchor':'end',fill:'#475569'},speed);st.textContent=s+' yd/s';
  }
  for(let i=1;i<d.end;i++)el('line',{x1:tx(d.time[i-1]),y1:sy(d.vy[i-1]),x2:tx(d.time[i]),y2:sy(d.vy[i]),stroke:'#176b87','stroke-width':1.8,'stroke-dasharray':'4,3'},speed);
  for(let i=1;i<d.s.length;i++)el('line',{x1:tx(d.time[i-1]),y1:sy(d.s[i-1]),x2:tx(d.time[i]),y2:sy(d.s[i]),stroke:'#2c3e50','stroke-width':2.3},speed);
  let peakIdx=0;for(let i=1;i<d.s.length;i++)if(d.s[i]>d.s[peakIdx])peakIdx=i;
  el('circle',{cx:tx(d.time[peakIdx]),cy:sy(d.s[peakIdx]),r:5.5,fill:'#b43b45',stroke:'#fff','stroke-width':1.2},speed);
  for(let tm=0;tm<=d.time.at(-1);tm+=2){const tt=el('text',{x:tx(tm),y:320,'font-size':11,'text-anchor':'middle',fill:'#475569'},speed);tt.textContent=tm+' s'}
}
function toggleLang(){
  lang=lang==='tr'?'en':'tr';
  const t=i18n[lang];
  document.getElementById('t-title').textContent=t.title;
  document.getElementById('t-sub').textContent=t.sub;
  document.getElementById('l-year').textContent=t.lyear;
  document.getElementById('l-pos').textContent=t.lpos;
  document.getElementById('l-sort').textContent=t.lsort;
  document.getElementById('l-player').textContent=t.lplayer;
  fYear.options[0].textContent=t.allYears;
  fPos.options[0].textContent=t.allPos;
  fSort.options[0].textContent=t.sName;
  fSort.options[1].textContent=t.sLat;
  fSort.options[2].textContent=t.sWhole;
  fSort.options[3].textContent=t.sShift;
  document.getElementById('k1-t').textContent=t.k1t;
  document.getElementById('k2-t').textContent=t.k2t;
  document.getElementById('k3-t').textContent=t.k3t;
  document.getElementById('k4-t').textContent=t.k4t;
  document.getElementById('c1-t').textContent=t.c1t;
  document.getElementById('c2-t').textContent=t.c2t;
  document.getElementById('lg-1').textContent=t.lg1;
  document.getElementById('lg-2').textContent=t.lg2;
  document.getElementById('lg-3').textContent=t.lg3;
  document.getElementById('t-note').textContent=t.note;
  document.getElementById('s2-title').textContent=t.s2t;
  document.getElementById('th-1').textContent=t.th1;
  document.getElementById('th-2').textContent=t.th2;
  document.getElementById('th-3').textContent=t.th3;
  document.getElementById('th-4').textContent=t.th4;
  document.getElementById('th-5').textContent=t.th5;
  document.getElementById('s2-desc').textContent=t.s2d;
  draw();
}
[fYear,fPos,fSort].forEach(el=>el.addEventListener('change',applyFilters));
select.addEventListener('change',draw);
applyFilters();
</script></body></html>'''.replace("__DATA__", data)
    (OUT / "mirror_explorer.html").write_text(page, encoding="utf-8")
    print(json.dumps(report, indent=2))


main()

In [ ]:
display(Image(filename=str(WORK_DIR / 'reports/trenches/mirror_phase_story.png')))
display(Image(filename=str(WORK_DIR / 'reports/trenches/mirror_ranking_shift.png')))


## 6. Physical Invariance Checks and 400-Replicate Full-Refit Bootstrap

Seven synthetic checks verify exact invariance to coordinate translation, lateral
reflection, timestamp origin, and input row ordering, plus rejection of duplicate or
gapped frames and detection of synthetic reversals. The 2025 full-refit bootstrap
resamples whole training players within class and test players across `400`
replicates, refitting both play-context and player ridge models in every replicate.


In [ ]:
"""Physical-invariance checks and full training/test player-bootstrap uncertainty."""
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


ROOT = WORK_DIR
OUT = ROOT / "reports/trenches"


def invariant_checks():
    # Known lateral oscillation followed by a separate forward sprint.
    time = np.arange(0, 10.1, .1)
    phase = time <= 8
    x = np.where(phase, 45, 45 + (time - 8) * 4)
    y = np.where(phase, 27 + 2 * np.sin(2 * np.pi * time / 2), 27)
    speed = np.sqrt(np.gradient(x, time) ** 2 + np.gradient(y, time) ** 2)
    frame = pd.DataFrame({"time": pd.Timestamp("2025-01-01") + pd.to_timedelta(time, unit="s"),
                          "x": x, "y": y, "s": speed})
    original, reason, _ = measure(frame)
    assert reason is None and original["reversals"] >= 6
    translated = frame.copy()
    translated["x"] += 100
    translated["y"] -= 80
    reflected = frame.copy()
    reflected["y"] *= -1
    shifted_time = frame.copy()
    shifted_time["time"] += pd.Timedelta(days=10)
    for label, changed in [("coordinate_translation", translated), ("lateral_reflection", reflected),
                           ("timestamp_shift", shifted_time), ("input_row_order", frame.sample(frac=1, random_state=7))]:
        actual, reason, _ = measure(changed)
        assert reason is None, label
        for column in ["lateral_speed_p75", "reversal_recovery", "mirror_duration", "reversals"]:
            assert np.isclose(actual[column], original[column], atol=1e-9), (label, column)
    duplicate = pd.concat([frame, frame.iloc[[10]]])
    assert measure(duplicate)[1] == "invalid_time_or_short"
    gap = frame.drop(index=[20, 21, 22])
    assert measure(gap)[1] == "invalid_time_or_short"
    return ["translation_invariance", "reflection_invariance", "time_origin_invariance", "row_order_invariance",
            "duplicate_timestamp_rejected", "missing_frame_gap_rejected", "synthetic_reversals_detected"]


def main():
    checks = invariant_checks()
    table, plays = load_data()
    train = table.loc[table.draft_year.lt(2025)].reset_index(drop=True)
    test = table.loc[table.draft_year.eq(2025)].reset_index(drop=True)
    train_p = plays.loc[plays.draft_year.lt(2025)].copy()
    test_p = plays.loc[plays.draft_year.eq(2025)].copy()
    num = ["down", "yards_to_go", "dropback_duration", "duration_sq"]
    cat = ["lined_up_position", "offense_formation"]
    grouped = {pid: frame for pid, frame in train_p.groupby("nfl_id")}
    rng = np.random.default_rng(2027)
    samples = []
    for replicate in range(400):
        # Preserve class counts while resampling whole players, never individual plays.
        train_sample = pd.concat([g.iloc[rng.integers(0, len(g), len(g))] for _, g in train.groupby("draft_year")])
        test_sample = test.iloc[rng.integers(0, len(test), len(test))].copy()
        sampled_plays = pd.concat([grouped[pid] for pid in train_sample.nfl_id], ignore_index=True)
        context = make_pipeline(preprocessing(num, cat), LogisticRegression(C=1, max_iter=2000))
        context.fit(sampled_plays[num + cat], sampled_plays.pressure)
        train_residual = pd.Series(100 * (train_p.pressure.to_numpy() - context.predict_proba(train_p[num + cat])[:, 1]),
                                   index=train_p.nfl_id).groupby(level=0).mean()
        test_residual = pd.Series(100 * (test_p.pressure.to_numpy() - context.predict_proba(test_p[num + cat])[:, 1]),
                                  index=test_p.nfl_id).groupby(level=0).mean()
        train_y = train_sample.nfl_id.map(train_residual)
        test_y = test_sample.nfl_id.map(test_residual)
        record = {"replicate": replicate}
        for label, columns in [("classic", CLASSIC), ("mirror", CLASSIC + MIRROR),
                               ("whole_drill", CLASSIC + ["whole_drill_speed"])]:
            model = player_model(columns).fit(train_sample[columns + ["nfl_position"]], train_y)
            prediction = model.predict(test_sample[columns + ["nfl_position"]])
            record[label + "_mae"] = float(np.mean(np.abs(test_y - prediction)))
        record["mirror_improvement"] = record["classic_mae"] - record["mirror_mae"]
        record["whole_drill_improvement"] = record["classic_mae"] - record["whole_drill_mae"]
        samples.append(record)
    results = pd.DataFrame(samples)
    results.to_csv(OUT / "mirror_full_refit_bootstrap.csv", index=False)
    summary = {"status": "passed", "invariant_checks": checks, "bootstrap_replicates": len(results),
               "test_year": 2025,
               "method": "Stratified training-player and test-player resampling, refitting context and player models each replicate.",
               "intervals_pp": {column: results[column].quantile([.025, .975]).tolist()
                                for column in ["mirror_improvement", "whole_drill_improvement"]},
               "limitations": "Exploratory percentile intervals with 400 replicates; conditional on observed games per resampled player; not adjusted for research multiplicity."}
    (OUT / "mirror_validation.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(json.dumps(summary, indent=2))


main()

## Summary & Practical Takeaways for NFL Front Offices

1. **Universal Exit-Sprint Contamination (`470 / 470` OL Recordings):** Across `Pass-Pro Mirror` (`113/113`), `5-Yard Wave` (`118/118`), and `Pass Rush Drop` (`239/239`), maximum smoothed speed always occurs in the finishing sprint rather than the lateral or pass-set phase.
2. **Repeatability Explains Attenuation:** In two-attempt `Pass Rush Drop` data (`n = 118` paired OL), exit sprint speed repeats at `r = +0.510`, whereas reactive pass-set lateral speed repeats at `r = +0.104`, bounding downstream game correlations.
3. **Honest Out-of-Time Validation:** Removing the exit sprint prevents `whole_drill_speed` from degrading 2025 pass-protection predictions, but single-rep mirror features do not achieve statistically significant out-of-time improvements over classic Combine measurements (`2025` full-refit 95% CI: `[−0.264, +0.353]` percentage points).

Sources: [NFL Big Data Bowl 2027](https://www.kaggle.com/competitions/nfl-big-data-bowl-2027),
[Offensive Line Combine Drill Coaching Guide](https://coachesinsider.com/football/offensive-linemen-specific-combine-drills/).
